In [ ]:
%number_of_workers 80


In [ ]:
%stop_session


# Pack 绝缘异常案例 EDA

目标：检查已知绝缘异常案例在云端是否存在可复现的绝缘电阻下降、
绝缘告警、正负母线不平衡或同场站 peer 差异，为后续模型逻辑提供证据。

本 notebook 只查询到云端可定位的子系统/Rack，不把现场 Pack 编号当成云端定位条件。
阈值和候选点均为 EDA 规则，不是正式告警逻辑。


## 案例覆盖与假设

| case_id | 云端目标 | 时间窗口 | 说明 |
|---|---|---|---|
| K0315_PACK_INSULATION_ABNORMAL | K0315.ESS43 / RBMS406 | 2026-04-01 ~ 2026-05-12 | Box 架构；RBMS为主，BBMS + PCS为辅助；Pack2只作现场信息 |
| K0056_PACK_INSULATION_ABNORMAL | ZHCN.ESS22 / RBMS204 | 2022-12-01 ~ 2023-04-18 | 故障日期不明确，扩大回看；返厂 NTF |
| K0043_PACK_INSULATION_ABNORMAL | WCCN.ESS22 / RBMS204 | 2023-01-01 ~ 2023-05-31 | “22-2-4 Rack”按子系统22、Rack204解析 |
| K0357_PACK_INSULATION_ABNORMAL | K0357.ESS16 / RBMS401 | 2026-07-15 ~ 2026-08-15 | Box 架构；RBMS为主，BBMS + PCS为辅助；Pack2只作现场信息 |

统一策略：所有纳入案例均使用目标Rack的RBMS作为主分析；Box案例统一增加BBMS和PCS辅助分析；
同子系统其他Rack作为RBMS peer，Pack编号只作现场备注，不进入云端查询条件。

暂不纳入：两条工厂案例；K0259“401簇”缺少子系统，无法可靠映射
`asset_storage_key`；其余信息不全的工厂/半成品案例。


In [ ]:
from pyspark.sql import functions as F

import boto3
import json

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (15, 8)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


In [ ]:
# site_code 和 device_type 已根据 site_configuration_modified.csv 解析。
CONFIG = json.loads(r'''{
  "s3_output_uri": "s3://engineeringbigdata/PILOT/EnergyStorageOperationData/EnergyStorageAnalysisResult/hannah.zhi/cbb_models/pack_cbb/eda/insulation_abnormal",
  "s3_bucket": "engineeringbigdata",
  "s3_prefix": "PILOT/EnergyStorageOperationData/EnergyStorageAnalysisResult/hannah.zhi/cbb_models/pack_cbb/eda/insulation_abnormal",
  "database": "pd_galileo_prod",
  "table_prefix": "energystorageoperationdata_prod_galileo_energystorageoperationdata",
  "cases": [
    {
      "case_id": "K0315_PACK_INSULATION_ABNORMAL",
      "site_num": "K-0315",
      "site_code": "K0315",
      "device_type": "box",
      "subsystem": 43,
      "rack": 406,
      "pack_note": "2号Pack（仅保留现场定位信息）",
      "storage": "K0315.ESS43",
      "component": "RBMS406",
      "start_day": "2026-04-01",
      "end_day": "2026-05-12",
      "window_note": "文档仅给出5/12且文件形成于2026年，暂按2026-05-12作为结束日，向前取约6周",
      "rca": "Pack内部冷却液泄漏，水冷密封失效"
    },
    {
      "case_id": "K0056_PACK_INSULATION_ABNORMAL",
      "site_num": "K-0056",
      "site_code": "ZHCN",
      "device_type": "cabinet",
      "subsystem": 22,
      "rack": 204,
      "pack_note": "8号Pack（云端不按Pack查询）",
      "storage": "ZHCN.ESS22",
      "component": "RBMS204",
      "start_day": "2022-12-01",
      "end_day": "2023-04-18",
      "window_note": "故障日期不明确，以2023-04-18返厂不复现记录为结束日，向前扩大窗口",
      "rca": "NTF，返厂后故障不复现"
    },
    {
      "case_id": "K0043_PACK_INSULATION_ABNORMAL",
      "site_num": "K-0043",
      "site_code": "WCCN",
      "device_type": "cabinet",
      "subsystem": 22,
      "rack": 204,
      "pack_note": "1号Pack（云端不按Pack查询）",
      "storage": "WCCN.ESS22",
      "component": "RBMS204",
      "start_day": "2023-01-01",
      "end_day": "2023-05-31",
      "window_note": "文档未给日期，结合相邻案例时间暂取2023年上半年；22-2-4 Rack按22号子系统、204 Rack解析",
      "rca": "PC片背胶吸附酒精后形成对箱体导通"
    },
    {
      "case_id": "K0357_PACK_INSULATION_ABNORMAL",
      "site_num": "K-0357",
      "site_code": "K0357",
      "device_type": "box",
      "subsystem": 16,
      "rack": 401,
      "pack_note": "2号Pack（仅保留现场定位信息）",
      "storage": "K0357.ESS16",
      "component": "RBMS401",
      "start_day": "2026-07-15",
      "end_day": "2026-08-15",
      "window_note": "2026-08-15发生故障，向前取一个月观察绝缘电阻变化",
      "rca": "待补充"
    }
  ]
}''')

S3_OUTPUT = CONFIG["s3_output_uri"].rstrip("/")
S3_BUCKET = CONFIG["s3_bucket"]
S3_PREFIX = CONFIG["s3_prefix"]
DATABASE = CONFIG["database"]
TABLE_PREFIX = CONFIG["table_prefix"]
CASES = CONFIG["cases"]
RBMS_CASES = CASES
BOX_CASES = [c for c in CASES if c["device_type"] == "box"]

print("all cases:", len(CASES))
print("RBMS cases:", len(RBMS_CASES))
print("box cases:", len(BOX_CASES))
print("S3 output:", S3_OUTPUT)


## 1. Rack级案例：RBMS 绝缘电阻、告警和运行状态

所有已纳入案例都具备子系统和Rack信息，因此统一使用RBMS作为主分析路径。
`rb_isoris` 是连续绝缘电阻（MΩ）；告警/故障字段用于确认云端是否记录到
现场现象；电流、SOC 和 Bat/PCS 电压用于判断异常是否依赖运行状态。


In [ ]:
rbms_table = f"{DATABASE}.{TABLE_PREFIX}_rbms_1s"


def load_rbms_case(case):
    '''查询一个可定位到RBMS的目标Rack。'''
    sql = f'''
    SELECT
        site_code,
        asset_storage_key,
        asset_component_key,
        timestamp,
        rb_a,
        rb_soc,
        rb_chast,
        rb_isoris,
        rb_isodctmod,
        rb_udisown,
        rb_udisoflt,
        rb_evthwiso,
        rb_grdflt,
        rb_volbat,
        rb_volpcs
    FROM {rbms_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
      AND asset_storage_key = '{case["storage"]}'
      AND asset_component_key = '{case["component"]}'
    '''
    return spark.sql(sql).cache()


rbms_data = {}
for case in RBMS_CASES:
    case_id = case["case_id"]
    print("loading:", case_id)
    rbms_data[case_id] = load_rbms_case(case)
    print("row count:", rbms_data[case_id].count())


In [ ]:
# 基础统计、日汇总和直接告警点。
for case in RBMS_CASES:
    case_id = case["case_id"]
    df = rbms_data[case_id]
    print("\n=====", case_id, "=====")

    df.select(
        "rb_isoris", "rb_a", "rb_soc", "rb_volbat", "rb_volpcs",
        "rb_udisown", "rb_udisoflt", "rb_evthwiso", "rb_grdflt",
    ).describe().show()

    daily_summary = (
        df
        .groupBy(
            "asset_storage_key",
            "asset_component_key",
            F.to_date("timestamp").alias("day"),
        )
        .agg(
            F.count("*").alias("row_count"),
            F.min("rb_isoris").alias("min_isoris_mohm"),
            F.expr("percentile_approx(rb_isoris, 0.1)").alias("isoris_p10_mohm"),
            F.expr("percentile_approx(rb_isoris, 0.5)").alias("isoris_p50_mohm"),
            F.avg("rb_a").alias("avg_current"),
            F.avg("rb_soc").alias("avg_soc"),
            F.avg("rb_volbat").alias("avg_bat_voltage"),
            F.avg("rb_volpcs").alias("avg_pcs_voltage"),
            F.max("rb_udisown").alias("iso_warning"),
            F.max("rb_udisoflt").alias("iso_fault"),
            F.max("rb_evthwiso").alias("iso_hardware_fault"),
            F.max("rb_grdflt").alias("ground_fault"),
        )
        .orderBy("day")
    )
    daily_summary.show(200, False)
    daily_summary.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/daily_summary"
    )

    alarm_events = df.filter(
        (F.coalesce(F.col("rb_udisown"), F.lit(0)) > 0)
        | (F.coalesce(F.col("rb_udisoflt"), F.lit(0)) > 0)
        | (F.coalesce(F.col("rb_evthwiso"), F.lit(0)) > 0)
        | (F.coalesce(F.col("rb_grdflt"), F.lit(0)) > 0)
    )
    print(case_id, "alarm rows:", alarm_events.count())
    alarm_events.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/alarm_events"
    )


In [ ]:
# 每个RBMS案例输出一张2x2图；1秒数据先聚合到分钟。
for case in RBMS_CASES:
    case_id = case["case_id"]
    plot_df = (
        rbms_data[case_id]
        .groupBy(F.window("timestamp", "1 minute").alias("minute"))
        .agg(
            F.avg("rb_isoris").alias("insulation_resistance"),
            F.max("rb_udisown").alias("iso_warning"),
            F.max("rb_udisoflt").alias("iso_fault"),
            F.max("rb_evthwiso").alias("iso_hardware_fault"),
            F.max("rb_grdflt").alias("ground_fault"),
            F.avg("rb_volbat").alias("bat_voltage"),
            F.avg("rb_volpcs").alias("pcs_voltage"),
            F.avg("rb_a").alias("current"),
            F.avg("rb_soc").alias("soc"),
        )
        .select(F.col("minute.start").alias("timestamp"), "*")
        .drop("minute")
        .orderBy("timestamp")
        .toPandas()
    )

    fig, axes = plt.subplots(2, 2, figsize=(17, 10))

    valid_iso = plot_df[plot_df["insulation_resistance"] > 0]
    axes[0, 0].plot(
        valid_iso["timestamp"], valid_iso["insulation_resistance"]
    )
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_title("Insulation resistance (MOhm, log scale)")

    for field in [
        "iso_warning", "iso_fault", "iso_hardware_fault", "ground_fault"
    ]:
        axes[0, 1].step(plot_df["timestamp"], plot_df[field], label=field)
    axes[0, 1].legend()
    axes[0, 1].set_title("Insulation / ground flags")

    axes[1, 0].plot(
        plot_df["timestamp"], plot_df["bat_voltage"], label="Bat voltage"
    )
    axes[1, 0].plot(
        plot_df["timestamp"], plot_df["pcs_voltage"], label="PCS voltage"
    )
    axes[1, 0].legend()
    axes[1, 0].set_title("Rack voltage")

    scatter_df = plot_df.dropna(subset=["current", "insulation_resistance"])
    scatter_df = scatter_df[scatter_df["insulation_resistance"] > 0]
    axes[1, 1].scatter(
        scatter_df["current"],
        scatter_df["insulation_resistance"],
        s=8,
        alpha=0.3,
    )
    axes[1, 1].set_yscale("log")
    axes[1, 1].set_xlabel("Rack current (A)")
    axes[1, 1].set_ylabel("Insulation resistance (MOhm)")

    fig.suptitle(case_id)
    fig.tight_layout()

    local_path = "/tmp/{}_rbms_insulation_eda.jpg".format(case_id)
    plt.savefig(local_path, dpi=300)

    s3_bucket = S3_BUCKET
    s3_key = "{}/figures/{}_rbms_insulation_eda.jpg".format(
        S3_PREFIX, case_id
    )
    s3 = boto3.client("s3")
    try:
        s3.upload_file(local_path, s3_bucket, s3_key)
        print("文件成功上传到 S3：s3://{}/{}".format(s3_bucket, s3_key))
    except Exception as exc:
        print("文件上传失败：{}".format(exc))

%matplot plt


In [ ]:
# 同子系统其他 Rack 做 peer 对比；p01仅作为低值候选点。
for case in RBMS_CASES:
    case_id = case["case_id"]
    peer_sql = f'''
    SELECT
        asset_component_key,
        timestamp,
        rb_isoris,
        rb_udisown,
        rb_udisoflt,
        rb_evthwiso,
        rb_grdflt,
        rb_volbat,
        rb_volpcs,
        rb_a,
        rb_soc
    FROM {rbms_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
      AND asset_storage_key = '{case["storage"]}'
    '''

    peer_summary = (
        spark.sql(peer_sql)
        .filter(F.col("rb_isoris") > 0)
        .groupBy("asset_component_key")
        .agg(
            F.min("rb_isoris").alias("min_isoris_mohm"),
            F.expr("percentile_approx(rb_isoris, 0.01)").alias("isoris_p01_mohm"),
            F.expr("percentile_approx(rb_isoris, 0.1)").alias("isoris_p10_mohm"),
            F.expr("percentile_approx(rb_isoris, 0.5)").alias("isoris_p50_mohm"),
            F.max("rb_udisown").alias("iso_warning"),
            F.max("rb_udisoflt").alias("iso_fault"),
            F.max("rb_grdflt").alias("ground_fault"),
            F.count("*").alias("valid_row_count"),
        )
        .orderBy("isoris_p01_mohm")
    )
    print("peer summary:", case_id)
    peer_summary.show(200, False)
    peer_summary.coalesce(1).write.mode("append").option(
        "header", True
    ).csv(f"{S3_OUTPUT}/tables/{case_id}/peer_summary")

    target_valid = rbms_data[case_id].filter(F.col("rb_isoris") > 0)
    threshold = target_valid.agg(
        F.expr("percentile_approx(rb_isoris, 0.01)").alias("isoris_p01")
    ).first()["isoris_p01"]

    if threshold is None:
        print(case_id, "没有有效绝缘电阻数据，跳过低值候选点")
        continue

    candidates = rbms_data[case_id].filter(
        ((F.col("rb_isoris") > 0) & (F.col("rb_isoris") <= threshold))
        | (F.coalesce(F.col("rb_udisown"), F.lit(0)) > 0)
        | (F.coalesce(F.col("rb_udisoflt"), F.lit(0)) > 0)
        | (F.coalesce(F.col("rb_grdflt"), F.lit(0)) > 0)
    )
    print(case_id, "isoris p01:", threshold)
    candidates.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/candidate_points"
    )


## 2. Box 案例：BBMS + PCS

K0315和K0357均为Box场站：RBMS负责Rack级主分析，BBMS用于观察子系统绝缘/接地标志，
PCS用于观察绝缘故障和正负直流母线电压。BBMS/PCS只作辅助，不能替代RBMS的Rack级定位。


In [ ]:
bbms_table = f"{DATABASE}.{TABLE_PREFIX}_bbms_1s"
pcs_table = f"{DATABASE}.{TABLE_PREFIX}_pcs_1s"


box_bbms_data = {}
box_pcs_data = {}

for case in BOX_CASES:
    case_id = case["case_id"]
    bbms_sql = f'''
    SELECT
        site_code,
        asset_storage_key,
        asset_component_key,
        timestamp,
        bb_v,
        bb_a,
        bb_soc,
        bb_chast,
        bb_rckisonot,
        bb_rckisoflt,
        bb_isohwflt,
        bb_fltgrd
    FROM {bbms_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
      AND asset_storage_key = '{case["storage"]}'
    '''

    pcs_sql = f'''
    SELECT
        site_code,
        asset_storage_key,
        asset_component_key,
        timestamp,
        pcs_batsoc,
        pcs_dcv,
        pcs_dcvp,
        pcs_dcvn,
        pcs_isoflt,
        pcs_mpnbusvolimblance,
        pcs_mposbus2peov,
        pcs_mnegbus2peov
    FROM {pcs_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
      AND asset_storage_key = '{case["storage"]}'
    '''

    box_bbms_data[case_id] = spark.sql(bbms_sql).cache()
    box_pcs_data[case_id] = (
        spark.sql(pcs_sql)
        .withColumn(
            "bus_balance_diff",
            F.abs(F.abs(F.col("pcs_dcvp")) - F.abs(F.col("pcs_dcvn"))),
        )
        .cache()
    )
    print(case_id, "BBMS rows:", box_bbms_data[case_id].count())
    print(case_id, "PCS rows:", box_pcs_data[case_id].count())


In [ ]:
# 保存日汇总和直接告警点。
for case in BOX_CASES:
    case_id = case["case_id"]
    bbms_df = box_bbms_data[case_id]
    pcs_df = box_pcs_data[case_id]

    print("\n=====", case_id, "BBMS =====")
    bbms_df.select(
        "bb_v", "bb_a", "bb_soc", "bb_rckisonot",
        "bb_rckisoflt", "bb_isohwflt", "bb_fltgrd",
    ).describe().show()

    bbms_daily = (
        bbms_df
        .groupBy(
            "asset_storage_key", "asset_component_key",
            F.to_date("timestamp").alias("day"),
        )
        .agg(
            F.count("*").alias("row_count"),
            F.avg("bb_v").alias("avg_voltage"),
            F.avg("bb_a").alias("avg_current"),
            F.avg("bb_soc").alias("avg_soc"),
            F.max("bb_rckisonot").alias("iso_notice"),
            F.max("bb_rckisoflt").alias("iso_fault"),
            F.max("bb_isohwflt").alias("iso_hardware_fault"),
            F.max("bb_fltgrd").alias("ground_fault"),
        )
        .orderBy("day")
    )
    bbms_daily.show(100, False)
    bbms_daily.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/bbms_daily_summary"
    )

    pcs_daily = (
        pcs_df
        .groupBy(
            "asset_storage_key", "asset_component_key",
            F.to_date("timestamp").alias("day"),
        )
        .agg(
            F.count("*").alias("row_count"),
            F.avg("pcs_dcv").alias("avg_dc_voltage"),
            F.avg("pcs_dcvp").alias("avg_positive_bus_voltage"),
            F.avg("pcs_dcvn").alias("avg_negative_bus_voltage"),
            F.expr("percentile_approx(bus_balance_diff, 0.9)").alias("balance_diff_p90"),
            F.max("bus_balance_diff").alias("max_balance_diff"),
            F.max("pcs_isoflt").alias("iso_fault"),
            F.max("pcs_mpnbusvolimblance").alias("pn_bus_imbalance"),
            F.max("pcs_mposbus2peov").alias("positive_bus_to_pe"),
            F.max("pcs_mnegbus2peov").alias("negative_bus_to_pe"),
        )
        .orderBy("day")
    )
    pcs_daily.show(100, False)
    pcs_daily.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/pcs_daily_summary"
    )

    bbms_alarm_events = bbms_df.filter(
        (F.coalesce(F.col("bb_rckisonot"), F.lit(0)) > 0)
        | (F.coalesce(F.col("bb_rckisoflt"), F.lit(0)) > 0)
        | (F.coalesce(F.col("bb_isohwflt"), F.lit(0)) > 0)
        | (F.coalesce(F.col("bb_fltgrd"), F.lit(0)) > 0)
    )
    pcs_alarm_events = pcs_df.filter(
        (F.coalesce(F.col("pcs_isoflt"), F.lit(0)) > 0)
        | (F.coalesce(F.col("pcs_mpnbusvolimblance"), F.lit(0)) > 0)
        | (F.coalesce(F.col("pcs_mposbus2peov"), F.lit(0)) > 0)
        | (F.coalesce(F.col("pcs_mnegbus2peov"), F.lit(0)) > 0)
    )
    print(case_id, "BBMS alarm rows:", bbms_alarm_events.count())
    print(case_id, "PCS alarm rows:", pcs_alarm_events.count())
    bbms_alarm_events.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/bbms_alarm_events"
    )
    pcs_alarm_events.write.mode("append").parquet(
        f"{S3_OUTPUT}/tables/{case_id}/pcs_alarm_events"
    )


In [ ]:
# 每个Box案例输出一张BBMS + PCS综合图；1秒数据先聚合到分钟。
for case in BOX_CASES:
    case_id = case["case_id"]

    bbms_plot = (
        box_bbms_data[case_id]
        .groupBy(F.window("timestamp", "1 minute").alias("minute"))
        .agg(
            F.avg("bb_v").alias("voltage"),
            F.avg("bb_a").alias("current"),
            F.max("bb_rckisonot").alias("iso_notice"),
            F.max("bb_rckisoflt").alias("iso_fault"),
            F.max("bb_isohwflt").alias("iso_hardware_fault"),
            F.max("bb_fltgrd").alias("ground_fault"),
        )
        .select(F.col("minute.start").alias("timestamp"), "*")
        .drop("minute")
        .orderBy("timestamp")
        .toPandas()
    )

    pcs_plot = (
        box_pcs_data[case_id]
        .groupBy(F.window("timestamp", "1 minute").alias("minute"))
        .agg(
            F.avg("pcs_dcvp").alias("positive_bus_voltage"),
            F.avg("pcs_dcvn").alias("negative_bus_voltage"),
            F.avg("bus_balance_diff").alias("bus_balance_diff"),
            F.max("pcs_isoflt").alias("iso_fault"),
            F.max("pcs_mpnbusvolimblance").alias("pn_bus_imbalance"),
            F.max("pcs_mposbus2peov").alias("positive_bus_to_pe"),
            F.max("pcs_mnegbus2peov").alias("negative_bus_to_pe"),
        )
        .select(F.col("minute.start").alias("timestamp"), "*")
        .drop("minute")
        .orderBy("timestamp")
        .toPandas()
    )

    fig, axes = plt.subplots(2, 2, figsize=(17, 10))

    axes[0, 0].plot(
        bbms_plot["timestamp"], bbms_plot["voltage"], label="BBMS voltage"
    )
    axes[0, 0].plot(
        bbms_plot["timestamp"], bbms_plot["current"], label="BBMS current"
    )
    axes[0, 0].legend()
    axes[0, 0].set_title("BBMS operating state")

    for field in [
        "iso_notice", "iso_fault", "iso_hardware_fault", "ground_fault"
    ]:
        axes[0, 1].step(
            bbms_plot["timestamp"], bbms_plot[field], label=field
        )
    axes[0, 1].legend()
    axes[0, 1].set_title("BBMS insulation / ground flags")

    axes[1, 0].plot(
        pcs_plot["timestamp"],
        pcs_plot["positive_bus_voltage"],
        label="positive bus",
    )
    axes[1, 0].plot(
        pcs_plot["timestamp"],
        pcs_plot["negative_bus_voltage"],
        label="negative bus",
    )
    axes[1, 0].plot(
        pcs_plot["timestamp"],
        pcs_plot["bus_balance_diff"],
        label="balance diff",
    )
    axes[1, 0].legend()
    axes[1, 0].set_title("PCS DC bus voltage")

    for field in [
        "iso_fault", "pn_bus_imbalance",
        "positive_bus_to_pe", "negative_bus_to_pe",
    ]:
        axes[1, 1].step(
            pcs_plot["timestamp"], pcs_plot[field], label=field
        )
    axes[1, 1].legend()
    axes[1, 1].set_title("PCS insulation / bus flags")

    fig.suptitle(case_id)
    fig.tight_layout()

    local_path = "/tmp/{}_box_insulation_eda.jpg".format(case_id)
    plt.savefig(local_path, dpi=300)

    s3_bucket = S3_BUCKET
    s3_key = "{}/figures/{}_box_insulation_eda.jpg".format(
        S3_PREFIX, case_id
    )
    s3 = boto3.client("s3")
    try:
        s3.upload_file(local_path, s3_bucket, s3_key)
        print("文件成功上传到 S3：s3://{}/{}".format(s3_bucket, s3_key))
    except Exception as exc:
        print("文件上传失败：{}".format(exc))

%matplot plt


In [ ]:
# 每个Box案例与同场站其他子系统做BBMS、PCS peer对比。
for case in BOX_CASES:
    case_id = case["case_id"]

    bbms_peer_sql = f'''
    SELECT
        asset_storage_key,
        asset_component_key,
        bb_v,
        bb_rckisonot,
        bb_rckisoflt,
        bb_isohwflt,
        bb_fltgrd
    FROM {bbms_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
    '''
    bbms_peer_summary = (
        spark.sql(bbms_peer_sql)
        .groupBy("asset_storage_key", "asset_component_key")
        .agg(
            F.avg("bb_v").alias("avg_voltage"),
            F.max("bb_rckisonot").alias("iso_notice"),
            F.max("bb_rckisoflt").alias("iso_fault"),
            F.max("bb_isohwflt").alias("iso_hardware_fault"),
            F.max("bb_fltgrd").alias("ground_fault"),
            F.count("*").alias("row_count"),
        )
        .orderBy("asset_storage_key")
    )
    bbms_peer_summary.show(200, False)
    bbms_peer_summary.coalesce(1).write.mode("append").option(
        "header", True
    ).csv(f"{S3_OUTPUT}/tables/{case_id}/bbms_peer_summary")

    pcs_peer_sql = f'''
    SELECT
        asset_storage_key,
        asset_component_key,
        pcs_dcvp,
        pcs_dcvn,
        pcs_isoflt,
        pcs_mpnbusvolimblance,
        pcs_mposbus2peov,
        pcs_mnegbus2peov
    FROM {pcs_table}
    WHERE event_date >= '{case["start_day"]}'
      AND event_date <= '{case["end_day"]}'
      AND site_code = '{case["site_code"]}'
    '''
    pcs_peer_summary = (
        spark.sql(pcs_peer_sql)
        .withColumn(
            "bus_balance_diff",
            F.abs(F.abs(F.col("pcs_dcvp")) - F.abs(F.col("pcs_dcvn"))),
        )
        .groupBy("asset_storage_key", "asset_component_key")
        .agg(
            F.expr("percentile_approx(bus_balance_diff, 0.5)").alias("balance_diff_p50"),
            F.expr("percentile_approx(bus_balance_diff, 0.9)").alias("balance_diff_p90"),
            F.max("bus_balance_diff").alias("max_balance_diff"),
            F.max("pcs_isoflt").alias("iso_fault"),
            F.max("pcs_mpnbusvolimblance").alias("pn_bus_imbalance"),
            F.max("pcs_mposbus2peov").alias("positive_bus_to_pe"),
            F.max("pcs_mnegbus2peov").alias("negative_bus_to_pe"),
            F.count("*").alias("row_count"),
        )
        .orderBy(F.desc("balance_diff_p90"))
    )
    pcs_peer_summary.show(200, False)
    pcs_peer_summary.coalesce(1).write.mode("append").option(
        "header", True
    ).csv(f"{S3_OUTPUT}/tables/{case_id}/pcs_peer_summary")


## 3. 首轮结果判读重点

- 先确认案例时间窗内是否有数据，以及告警时间是否覆盖现场失效时间。
- K0056：若 RBMS 未出现低绝缘或 peer 差异，应如实保留负结论，与返厂 NTF 对照。
- K0043：关注目标 Rack 的绝缘电阻是否长期低于同子系统其他 Rack，以及是否随电压/电流状态变化。
- K0315：重点确认 RBMS406 的 `rb_isoris` 是否在5月12日前下降，并与43号子系统其他 Rack 对比；BBMS/PCS只作辅助证据。
- K0357：重点确认 RBMS401 的 `rb_isoris` 是否在8月15日前逐步下降或突然降至约0.271 MΩ，并与16号子系统其他 Rack 对比；BBMS/PCS只作辅助证据。
- 若连续量与告警都无分离度，下一步优先补充现场准确发生时刻、PCS更细粒度对地电压字段或维修标签，不强行形成模型规则。
